In [1]:
%pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# IMPORTS
import os
import random
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

# SEED DEFINITION
SEED = 67
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# PATHS
DATA_DIR = "../data/KDDdataset"
MODEL_DIR = "../models"
ART_DIR = "../artifacts"

In [3]:
# PREPROCESSING
train_df = pd.read_csv(os.path.join(DATA_DIR, "KDDTrain.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "KDDTest.csv"))

# removing attack score feature
train_df = train_df.drop(columns=["attack score"])
test_df = test_df.drop(columns=["attack score"])

# VALIDATION SET (10% of the original training data)
train_df, val_df = train_test_split(
    train_df, test_size=0.1, random_state=SEED, stratify=train_df["label"]
)

# preserving the same column order in all notebooks
feature_names = train_df.drop(columns=["label", "attack"]).columns.to_numpy(dtype=str)
assert test_df.drop(columns=["label", "attack"]).columns.tolist() == feature_names.tolist()

# ORDINAL ENCODING
categ_feats = ["protocol_type", "service", "flag"]
cat_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
train_df[categ_feats] = cat_encoder.fit_transform(train_df[categ_feats])
val_df[categ_feats] = cat_encoder.transform(val_df[categ_feats])
test_df[categ_feats] = cat_encoder.transform(test_df[categ_feats])

# SPLITTING FEATURES (X) AND LABELS (y)
X_train_df = train_df.drop(columns=["label", "attack"]).values
y_train = train_df["label"].values
X_val_df = val_df.drop(columns=["label", "attack"]).values
y_val = val_df["label"].values
X_test_df = test_df.drop(columns=["label", "attack"]).values
y_test = test_df["label"].values

# MIN-MAX SCALING
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_df)
X_val = scaler.transform(X_val_df)
X_test = scaler.transform(X_test_df)

print(f"Shapes | Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print("Labels: 0 = normal, 1 = attack")


Shapes | Train: (113375, 41) | Val: (12598, 41) | Test: (22544, 41)
Labels: 0 = normal, 1 = attack


In [4]:
# RANDOM FOREST BASELINE
rf_model = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
rf_prec = precision_score(y_test, rf_pred, zero_division=0)
rf_rec = recall_score(y_test, rf_pred, zero_division=0)
rf_f1 = f1_score(y_test, rf_pred, zero_division=0)

print("=== RANDOM FOREST BASELINE ===")
print(f"Accuracy : {rf_acc:.4f} | Precision: {rf_prec:.4f} | Recall: {rf_rec:.4f} | F1: {rf_f1:.4f}")
rf_cm = confusion_matrix(y_test, rf_pred, labels=[0, 1])
print("Confusion matrix (rows = true, columns = predicted; order: normal, attack):")
print(rf_cm)


=== RANDOM FOREST BASELINE ===
Accuracy : 0.7794 | Precision: 0.9665 | Recall: 0.6345 | F1: 0.7661
Confusion matrix (rows = true, columns = predicted; order: normal, attack):
[[9429  282]
 [4691 8142]]


In [5]:
# DEEP NEURAL NETWORK

# device detection
__torch_device = None
def get_torch_device():
    global __torch_device
    if __torch_device is not None:
        return __torch_device
    if  torch.backends.mps.is_available():
        print('Using MPS device for hardware acceleration')
        __torch_device = 'mps'
    elif torch.cuda.is_available():
        print('Using CUDA device for hardware acceleration')
        __torch_device = 'cuda'
    else:
        print('Hardware acceleration not available, using CPU')
        __torch_device = 'cpu'
    return __torch_device

device = get_torch_device()

# DNN CLASS DEFINITION
class DNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

# DNN TRAINING
dnn_model = DNN(X_train.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(dnn_model.parameters(), lr=0.001)
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
epochs = 10
dnn_model.train()
for epoch in range(epochs):
    epoch_loss = 0
    for bx, by in train_loader:
        bx = bx.to(device)
        by = by.to(device)
        optimizer.zero_grad()
        loss = criterion(dnn_model(bx), by)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * bx.size(0)

# EVALUATION
dnn_model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    logits = dnn_model(X_test_tensor)
    dnn_preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy().squeeze()

dnn_acc = accuracy_score(y_test, dnn_preds)
dnn_prec = precision_score(y_test, dnn_preds, zero_division=0)
dnn_rec = recall_score(y_test, dnn_preds, zero_division=0)
dnn_f1 = f1_score(y_test, dnn_preds, zero_division=0)

print("=== PYTORCH DNN BASELINE ===")
print(f"Accuracy : {dnn_acc:.4f} | Precision: {dnn_prec:.4f} | Recall: {dnn_rec:.4f} | F1: {dnn_f1:.4f}")
dnn_cm = confusion_matrix(y_test, dnn_preds, labels=[0, 1])
print("Confusion matrix (rows = true, columns = predicted; order: normal, attack):")
print(dnn_cm)

Using MPS device for hardware acceleration
=== PYTORCH DNN BASELINE ===
Accuracy : 0.7925 | Precision: 0.9711 | Recall: 0.6550 | F1: 0.7824
Confusion matrix (rows = true, columns = predicted; order: normal, attack):
[[9461  250]
 [4427 8406]]


In [6]:
# SAVING EVERYTHING
os.makedirs(os.path.join(MODEL_DIR, "clean"), exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.joblib"))
joblib.dump(cat_encoder, os.path.join(MODEL_DIR, "cat_encoder.joblib"))
joblib.dump(rf_model, os.path.join(MODEL_DIR, "clean", "clean_rf_model.joblib"))
torch.save(dnn_model.state_dict(), os.path.join(MODEL_DIR, "clean", "clean_dnn_model.pth"))
np.savez(
    os.path.join(ART_DIR, "clean_data.npz"),
    X_train = X_train,
    y_train = y_train,
    X_val = X_val,
    y_val = y_val,
    X_test = X_test,
    y_test = y_test,
    feature_names = feature_names
)

print("Everything saved")

Everything saved
